In [75]:
import os
from pathlib import Path
import pandas as pd
from IPython.display import display
from pyspark.sql import SparkSession

In [76]:
# Locate the project root
current_dir = Path.cwd().resolve()

project_root = next(
    (
        path
        for path in (current_dir, *current_dir.parents)
        if (
            (path / "advanced" / "ETL").is_dir()
            and (path / "advanced" / "Starter_Sources").is_dir()
        )
    ),
    None
)

if project_root is None:
    raise FileNotFoundError("Project root could not be identified.")

In [ ]:
# Main project directories
advanced_dir = project_root / "advanced"

# Source directories
data_dir = advanced_dir / "data"
starter_sources_dir = advanced_dir / "Starter_Sources"

# ETL directories
etl_dir = advanced_dir / "ETL"
notebooks_dir = etl_dir / "notebooks"
warehouse_dir = etl_dir / "warehouse"

# Spark temporary storage
spark_temp_dir = etl_dir / ".spark-temp"


# Medallion warehouse directories

bronze_warehouse_dir = warehouse_dir / "bronze"


# Create directories
directory = bronze_warehouse_dir
directory.mkdir(parents=True, exist_ok=True)

In [78]:
# Java and Hadoop configuration

java_home = Path(
    r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.101-hotspot"
)

hadoop_home = Path(
    r"C:\hadoop\hadoop-3.5.0"
)

# Verify required files
required_files = [
    java_home / "bin" / "java.exe",
    hadoop_home / "bin" / "winutils.exe",
    hadoop_home / "bin" / "hadoop.dll",
    hadoop_home / "bin" / "hdfs.dll"
]

for file in required_files:
    if not file.exists():
        raise FileNotFoundError(f"Missing file: {file}")

# Configure environment variables
os.environ["JAVA_HOME"] = str(java_home)
os.environ["HADOOP_HOME"] = str(hadoop_home)

# Update PATH
os.environ["PATH"] = os.pathsep.join([
    str(java_home / "bin"),
    str(hadoop_home / "bin"),
    os.environ.get("PATH", "")
])

print("Java configured:", java_home)
print("Hadoop configured:", hadoop_home)
print("Hadoop utilities verified.")

Java configured: C:\Program Files\Eclipse Adoptium\jdk-17.0.20.101-hotspot
Hadoop configured: C:\hadoop\hadoop-3.5.0
Hadoop utilities verified.


In [79]:
warehouse_dir.mkdir(parents=True, exist_ok=True)
spark_temp_dir.mkdir(parents=True, exist_ok=True)

In [80]:
# Configure Spark temporary storage
os.environ["SPARK_LOCAL_DIRS"] = str(spark_temp_dir)
os.environ["TEMP"] = str(spark_temp_dir)
os.environ["TMP"] = str(spark_temp_dir)


In [81]:
# Define the persistent metastore location
metastore_dir = etl_dir / "metastore_db"

metastore_url = (
    f"jdbc:derby:{metastore_dir.as_posix()};create=true"
)

In [82]:
spark = (
    SparkSession.builder
    .appName("AI_Monitoring")
    .master("local[2]")
    .config("spark.local.dir", str(spark_temp_dir))
    .config("spark.sql.warehouse.dir", warehouse_dir.as_uri())
    .config("spark.sql.catalogImplementation", "hive")
    .config(
        "spark.hadoop.javax.jdo.option.ConnectionURL",
        metastore_url
    )
    .enableHiveSupport()
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

In [83]:
spark.sql(
    f"CREATE DATABASE IF NOT EXISTS bronze "
    f"LOCATION '{bronze_warehouse_dir.as_uri()}'"
)

spark.sql(
    f"CREATE DATABASE IF NOT EXISTS silver "
    f"LOCATION '{silver_warehouse_dir.as_uri()}'"
)

spark.sql(
    f"CREATE DATABASE IF NOT EXISTS gold "
    f"LOCATION '{gold_warehouse_dir.as_uri()}'"
)

DataFrame[]

In [84]:
display(spark.sql("SHOW DATABASES").toPandas())

,namespace
0,bronze
1,default
2,gold
3,silver


In [85]:
# Define source directories
source_directories = {
    "Detector Outputs": data_dir,
    "Reference Sources": starter_sources_dir
}

In [86]:
files_to_inspect = {
    "persons_worker1": data_dir / "persons_detected_worker1.csv",
    "persons_worker2": data_dir / "persons_detected_worker2.csv",
    "persons_legacy": data_dir / "persons_detected.csv",
    "plates": data_dir / "plates_detected.csv",
    "employees": starter_sources_dir / "employees.csv",
    "cameras": starter_sources_dir / "cameras.csv",
    "restricted_access": starter_sources_dir / "restricted_access.csv"
}

In [87]:
worker1_df = (spark.read
            .option('Header', True)
            .option('inferschema', False)
            .option('quote', '"')
            .option("escape", '"')
            .option("mode", 'FAILFAST')    
            .csv(str(data_dir / "persons_detected_worker1.csv"))        
            )

In [88]:
worker2_df = (spark.read
            .option('Header', True)
            .option('inferschema', False)
            .option('quote', '"')
            .option("escape", '"')
            .option("mode", 'FAILFAST')    
            .csv(str(data_dir / "persons_detected_worker2.csv"))        
            )

In [89]:
worker1_df.write \
    .format("parquet") \
    .mode("overwrite") \
    .saveAsTable("bronze.person_detections_worker1")

In [90]:
worker2_df.write \
    .format("parquet") \
    .mode("overwrite") \
    .saveAsTable("bronze.person_detections_worker2")

In [91]:
Plates_df = (spark.read
            .option("Header", True)
            .option("InferSchema", False)
            .option("mode", "FAILFAST")
            .option("qoute", '"')
            .option("escape", '"')
            .csv(str(data_dir / "plates_detected.csv"))
            
            
            )

In [92]:
Plates_df.write \
    .format("parquet") \
    .mode("overwrite") \
    .saveAsTable("bronze.plate_detections")

In [93]:
Employees_df = (spark.read
            .option("Header", True)
            .option("InferSchema", False)
            .option("mode", "FAILFAST")
            .option("qoute", '"')
            .option("escape", '"')
            .csv(str(starter_sources_dir / "employees.csv"))
                    )

In [94]:
Cameras_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", False)
    .option("mode", "FAILFAST")
    .load(str(starter_sources_dir / "cameras.csv"))
)

In [95]:
# Read restricted access reference data

Restricted_access_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", False)
    .option("mode", "FAILFAST")
    .load(str(starter_sources_dir / "restricted_access.csv"))
)

In [96]:
Plates_df.write \
    .format("parquet") \
    .mode("overwrite") \
    .saveAsTable("bronze.plate_detections")

In [97]:
Employees_df.write \
    .format("parquet") \
    .mode("overwrite") \
    .saveAsTable("bronze.employees")

In [98]:
Cameras_df.write \
    .format("parquet") \
    .mode("overwrite") \
    .saveAsTable("bronze.cameras")

In [99]:
Restricted_access_df.write \
    .format("parquet") \
    .mode("overwrite") \
    .saveAsTable("bronze.restricted_access")

In [100]:
display(
    spark.sql("SHOW TABLES IN bronze").toPandas()
)

,namespace,tableName,isTemporary
0,bronze,cameras,False
1,bronze,employees,False
2,bronze,person_detections_worker1,False
3,bronze,person_detections_worker2,False
4,bronze,plate_detections,False
5,bronze,restricted_access,False


In [107]:
import sys
import importlib

# Add ETL directory to Python path
etl_dir = warehouse_dir.parent

if str(etl_dir) not in sys.path:
    sys.path.insert(0, str(etl_dir))

# Import and reload BronzeIngestion
import spark_pipeline.bronze.bronze_ingestion as bronze_module

importlib.reload(bronze_module)

BronzeIngestion = bronze_module.BronzeIngestion

In [108]:
bronze = BronzeIngestion(
    spark=spark,
    data_dir=data_dir,
    starter_sources_dir=starter_sources_dir
)

In [111]:
bronze.run()

STARTING BRONZE INGESTION
Ingesting Worker 1...
Successfully written: bronze.person_detections_worker1
Ingesting Worker 2...
Successfully written: bronze.person_detections_worker2
Successfully ingested 2 workers.
Successfully written: bronze.plate_detections
Successfully written: bronze.employees
Successfully written: bronze.cameras
Successfully written: bronze.restricted_access
BRONZE INGESTION COMPLETED


In [112]:
bronze_tables = [
    "person_detections_worker1",
    "person_detections_worker2",
    "plate_detections",
    "employees",
    "cameras",
    "restricted_access"
]

results = []

for table_name in bronze_tables:

    df = spark.table(f"bronze.{table_name}")

    results.append({
        "Table": table_name,
        "Records": df.count(),
        "Columns": len(df.columns)
    })

bronze_inventory = pd.DataFrame(results)

display(bronze_inventory)

,Table,Records,Columns
0,person_detections_worker1,16859,7
1,person_detections_worker2,22397,7
2,plate_detections,17613,6
3,employees,100,2
4,cameras,10,5
5,restricted_access,20,2


In [114]:
print("Bronze Metastore URL:", metastore_url)
print("Bronze Warehouse:", warehouse_dir)

Bronze Metastore URL: jdbc:derby:D:/projects/AI_Monitoring_Project/advanced/ETL/metastore_db;create=true
Bronze Warehouse: D:\projects\AI_Monitoring_Project\advanced\ETL\warehouse


In [116]:
spark.stop()